<a href="https://colab.research.google.com/github/fernandosuarez89/fernandosuarez89.github.io/blob/main/SEPA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import math

def distance_haversine(lat1, lon1, lat2, lon2, radius=6371):
    """
    Calculates the distance between two points on Earth using the Haversine formula.

    Args:
        lat1 (float or str): Latitude of the first point in degrees.
        lon1 (float or str): Longitude of the first point in degrees.
        lat2 (float or str): Latitude of the second point in degrees.
        lon2 (float or str): Longitude of the second point in degrees.
        radius (float, optional): Radius of the Earth in kilometers. Defaults to 6371.

    Returns:
        float or None: Distance between the two points in the same unit as the radius,
                       or None if coordinates cannot be converted to float.
    """
    try:
        lat1 = float(lat1)
        lon1 = float(lon1)
        lat2 = float(lat2)
        lon2 = float(lon2)
    except (ValueError, TypeError):
        return None # Return None if conversion fails for any coordinate

    # Convert latitude and longitude to radians
    lat1_rad = math.radians(lat1)
    lon1_rad = math.radians(lon1)
    lat2_rad = math.radians(lat2)
    lon2_rad = math.radians(lon2)

    # Calculate differences
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad

    # Apply Haversine formula
    a = math.sin(dlat / 2)**2 + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    # Calculate distance
    distance = radius * c
    return distance

# Example usage:
lat1, lon1 = 40.7128, -74.0060  # New York
lat2, lon2 = 34.0522, -118.2437  # Los Angeles
distance = distance_haversine(lat1, lon1, lat2, lon2)
print(f"The distance between New York and Los Angeles is: {distance:.2f} km")

The distance between New York and Los Angeles is: 3935.75 km


In [ ]:
import requests
import zipfile
import os
import pandas as pd
from datetime import datetime, timedelta
import pytz


def clean_column_names(df, prefixes):
    new_columns = []
    for col in df.columns:
        cleaned_col = col
        for prefix in prefixes:
            if cleaned_col.startswith(prefix):
                cleaned_col = cleaned_col[len(prefix):]
                break
        new_columns.append(cleaned_col)
    df.columns = new_columns
    return df


sepas = {
    "lunes": "https://datos.produccion.gob.ar/dataset/6f47ec76-d1ce-4e34-a7e1-621fe9b1d0b5/resource/0a9069a9-06e8-4f98-874d-da5578693290/download/sepa_lunes.zip",
    "martes": "https://datos.produccion.gob.ar/dataset/6f47ec76-d1ce-4e34-a7e1-621fe9b1d0b5/resource/9dc06241-cc83-44f4-8e25-c9b1636b8bc8/download/sepa_martes.zip",
    "miercoles": "https://datos.produccion.gob.ar/dataset/6f47ec76-d1ce-4e34-a7e1-621fe9b1d0b5/resource/1e92cd42-4f94-4071-a165-62c4cb2ce23c/download/sepa_miercoles.zip",
    "jueves": "https://datos.produccion.gob.ar/dataset/6f47ec76-d1ce-4e34-a7e1-621fe9b1d0b5/resource/d076720f-a7f0-4af8-b1d6-1b99d5a90c14/download/sepa_jueves.zip",
    "viernes": "https://datos.produccion.gob.ar/dataset/6f47ec76-d1ce-4e34-a7e1-621fe9b1d0b5/resource/91bc072a-4726-44a1-85ec-4a8467aad27e/download/sepa_viernes.zip",
    "sabado": "https://datos.produccion.gob.ar/dataset/6f47ec76-d1ce-4e34-a7e1-621fe9b1d0b5/resource/b3c3da5d-213d-41e7-8d74-f23fda0a3c30/download/sepa_sabado.zip",
    "domingo": "https://datos.produccion.gob.ar/dataset/6f47ec76-d1ce-4e34-a7e1-621fe9b1d0b5/resource/f8e75128-515a-436e-bf8d-5c63a62f2005/download/sepa_domingo.zip"
}


# Define Buenos Aires timezone
buenos_aires_tz = pytz.timezone('America/Argentina/Buenos_Aires')

# Get current time in Buenos Aires
now_ba = datetime.now(buenos_aires_tz)

# Determine the date for the data to fetch
if now_ba.hour >= 15:  # 3 PM (15:00) cutoff
    target_date = now_ba.date()
elif now_ba.hour < 15:
    target_date = now_ba.date() - timedelta(days=1)

# Get the day of the week (0=Monday, 6=Sunday)
day_of_week_num = target_date.weekday()

day_names = [
    "lunes",
    "martes",
    "miercoles",
    "jueves",
    "viernes",
    "sabado",
    "domingo",
]

# Get the corresponding day name
target_day_name = day_names[day_of_week_num]

# Initialize a flag to control processing
should_process_zip = False

# Ensure zip_file_name is always set
zip_file_name = f"sepa_{target_day_name}.zip"

# First attempt to get the URL
current_zip_url = sepas.get(target_day_name)

if not current_zip_url:
    print(f"Error: No ZIP URL found for {target_day_name} in the URL dictionary.")
    if os.path.exists(zip_file_name):
        print(f"However, ZIP file '{zip_file_name}' found locally. Attempting to use local file.")
        should_process_zip = True # Trust existing file for now, will validate before extraction
    else:
        print(f"And '{zip_file_name}' not found locally. Skipping file extraction and processing.")
else: # current_zip_url is valid
    if os.path.exists(zip_file_name):
        print(f"ZIP file '{zip_file_name}' already exists locally. Checking validity...")
        try:
            # Attempt to open as a zip file to validate it
            with zipfile.ZipFile(zip_file_name, 'r') as zip_test:
                zip_test.testzip() # This method checks the integrity of the archive
            print(f"Local ZIP file '{zip_file_name}' is valid. Skipping download.")
            should_process_zip = True
        except zipfile.BadZipFile:
            print(f"Warning: Local file '{zip_file_name}' is corrupt or not a valid ZIP file. Deleting and attempting re-download.")
            os.remove(zip_file_name) # Delete corrupt file
            # Fall through to download logic below
        except Exception as e:
            print(f"Error validating local ZIP file '{zip_file_name}': {e}. Deleting and attempting re-download.")
            os.remove(zip_file_name)
            # Fall through to download logic below

    # If no valid local file was found (either didn't exist, or was corrupt and deleted), then download
    if not should_process_zip:
        # Add headers to mimic a browser request
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }

        try:
            print(f"Attempting to download '{zip_file_name}' from {current_zip_url}.")
            response = requests.get(current_zip_url, headers=headers)
            response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)

            with open(zip_file_name, 'wb') as f:
                f.write(response.content)

            print(f"ZIP file '{zip_file_name}' downloaded successfully.")
            should_process_zip = True # Set to true if download was successful
        except requests.exceptions.RequestException as e:
            print(f"Error downloading ZIP file: {e}")
            print(f"Please try to manually download '{zip_file_name}' from the website and upload it to your Colab environment.")
            should_process_zip = False # Download failed, do not process

# Now, the main processing block, proceed only if should_process_zip is True and the file exists
if should_process_zip and os.path.exists(zip_file_name):
    # Final verification of the ZIP file before extraction to catch issues with newly downloaded files
    try:
        with zipfile.ZipFile(zip_file_name, 'r') as zip_test_extract:
            zip_test_extract.testzip() # This also raises BadZipFile if corrupt
    except zipfile.BadZipFile:
        print(f"Critical Error: '{zip_file_name}' is not a valid ZIP file even after (re)download/local check. Skipping extraction and further processing.")
        if os.path.exists(zip_file_name):
            os.remove(zip_file_name)
            print(f"Removed corrupt file '{zip_file_name}'.")
        should_process_zip = False
    except Exception as e:
        print(f"Critical Error: An unexpected error occurred while verifying '{zip_file_name}': {e}. Skipping extraction and further processing.")
        if os.path.exists(zip_file_name):
            os.remove(zip_file_name)
            print(f"Removed problematic file '{zip_file_name}'.")
        should_process_zip = False

if should_process_zip:
    # Directory to extract the contents to
    extract_dir = target_date.strftime('%Y-%m-%d')
    os.makedirs(extract_dir, exist_ok=True)

    # Create a ZipFile object and extract all contents
    with zipfile.ZipFile(zip_file_name, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)

    print(f"Contents of '{zip_file_name}' extracted to '{os.path.abspath(extract_dir)}' successfully.")

    # Determine the actual directory containing the files to process
    # This handles cases where the main ZIP extracts into a single top-level folder
    actual_content_dir = extract_dir
    extracted_items = os.listdir(extract_dir)
    if len(extracted_items) == 1 and os.path.isdir(os.path.join(extract_dir, extracted_items[0])):
        actual_content_dir = os.path.join(extract_dir, extracted_items[0])

    # Now, nested_zip_search_dir should point to the directory that actually contains the zips to be extracted
    nested_zip_search_dir = actual_content_dir

    # List all files in the nested_zip_search_dir
    all_files_in_nested_dir = os.listdir(nested_zip_search_dir)

    # Filter for .zip files (these are the inner zips)
    nested_zip_files = [f for f in all_files_in_nested_dir if f.endswith('.zip')]

    print(f"Found {len(nested_zip_files)} nested ZIP files in '{nested_zip_search_dir}':")
    for nested_zip_file_name in nested_zip_files:
        nested_zip_file_path = os.path.join(nested_zip_search_dir, nested_zip_file_name)

        # Create a sub-directory for each nested zip's extraction to avoid clutter
        # Extract into a directory within actual_content_dir
        extraction_sub_dir = os.path.join(actual_content_dir, nested_zip_file_name.replace('.zip', ''))
        os.makedirs(extraction_sub_dir, exist_ok=True)

        try:
            with zipfile.ZipFile(nested_zip_file_path, 'r') as nested_zip_ref:
                nested_zip_ref.extractall(extraction_sub_dir)
            print(f"  - Extracted '{nested_zip_file_name}' to '{extraction_sub_dir}'")
        except zipfile.BadZipFile:
            print(f"  - Warning: '{nested_zip_file_name}' is not a valid ZIP file and could not be extracted.")
            if os.path.exists(nested_zip_file_path):
                os.remove(nested_zip_file_path)
                print(f"    Removed corrupt nested file '{nested_zip_file_name}'.")
        except Exception as e:
            print(f"  - Error extracting '{nested_zip_file_name}': {e}")

    print("Finished extracting all identified nested ZIP files.")

    csv_files_found = []
    search_dir = actual_content_dir # Scan for CSVs starting from the actual content directory

    # Walk through the directory and its subdirectories
    for root, dirs, files in os.walk(search_dir):
        for file in files:
            if file.endswith('.csv'):
                csv_files_found.append(os.path.join(root, file))

    print(f"Found {len(csv_files_found)} CSV files:")


    # Initialize lists to store categorized files and their sources
    productos_files = []
    sucursales_files = []
    comercio_files = []

    # Iterate through each file path in the csv_files_found list
    for file_path in csv_files_found:
        # Extract the base name of the file (e.g., 'productos.csv')
        file_name = os.path.basename(file_path)
        # Extract the directory path of the file (the parent directory)
        file_source = os.path.dirname(file_path)

        # Categorize the file based on its name and append to the corresponding list
        if 'productos.csv' in file_name:
            productos_files.append((file_path, file_source))
        elif 'sucursales.csv' in file_name:
            sucursales_files.append((file_path, file_source))
        elif 'comercio.csv' in file_name:
            comercio_files.append((file_path, file_source))
        else:
            print(f"Warning: Uncategorized CSV file found: {file_path}")

    print(f"Total productos files found: {len(productos_files)}")
    print(f"Total sucursales files found: {len(sucursales_files)}")
    print(f"Total comercio files found: {len(comercio_files)}")
else:
    print("Skipping file extraction and processing due to unresolvable issues (download failed or no valid local file found).")

ZIP file 'sepa_miercoles.zip' already exists locally. Checking validity...
Local ZIP file 'sepa_miercoles.zip' is valid. Skipping download.
Contents of 'sepa_miercoles.zip' extracted to '/content/2026-06-24' successfully.
Found 17 nested ZIP files in '2026-06-24/2026-06-24':
  - Extracted 'sepa_1_comercio-sepa-19_2026-06-24_09-05-10.zip' to '2026-06-24/2026-06-24/sepa_1_comercio-sepa-19_2026-06-24_09-05-10'
  - Extracted 'sepa_2_comercio-sepa-5_2026-06-24_01-05-07.zip' to '2026-06-24/2026-06-24/sepa_2_comercio-sepa-5_2026-06-24_01-05-07'
  - Extracted 'sepa_1_comercio-sepa-21_2026-06-24_09-05-10.zip' to '2026-06-24/2026-06-24/sepa_1_comercio-sepa-21_2026-06-24_09-05-10'
  - Extracted 'sepa_2_comercio-sepa-11_2026-06-24_01-05-07.zip' to '2026-06-24/2026-06-24/sepa_2_comercio-sepa-11_2026-06-24_01-05-07'
  - Extracted 'sepa_1_comercio-sepa-16_2026-06-24_09-05-10.zip' to '2026-06-24/2026-06-24/sepa_1_comercio-sepa-16_2026-06-24_09-05-10'
  - Extracted 'sepa_1_comercio-sepa-20_2026-06-24_0

In [ ]:
all_productos_dfs = []
for file_path, file_source in productos_files:
    try:
        df = pd.read_csv(file_path, sep='|', dtype={'id_comercio': str, 'id_bandera': str, 'id_sucursal': str}, low_memory=False)
        df['file_source'] = file_source
        all_productos_dfs.append(df)
    except Exception as e:
        print(f"Error loading {file_path}: {e}")

if all_productos_dfs:
    all_productos_df = pd.concat(all_productos_dfs, ignore_index=True)
    # Apply column cleaning
    productos_prefixes = ['productos_']
    all_productos_df = clean_column_names(all_productos_df, productos_prefixes)
else:
    print("No productos files were loaded.")

all_sucursales_dfs = []
for file_path, file_source in sucursales_files:
    try:
        # Specify 'id_comercio', 'id_bandera', and 'id_sucursal' as string type to avoid DtypeWarning
        df = pd.read_csv(file_path, sep='|', dtype={'id_comercio': str, 'id_bandera': str, 'id_sucursal': str}, low_memory=False)
        df['file_source'] = file_source
        all_sucursales_dfs.append(df)
    except Exception as e:
        print(f"Error loading {file_path}: {e}")

if all_sucursales_dfs:
    all_sucursales_df = pd.concat(all_sucursales_dfs, ignore_index=True)
    # Apply column cleaning
    sucursales_prefixes = ['sucursales_']
    all_sucursales_df = clean_column_names(all_sucursales_df, sucursales_prefixes)
else:
    print("No sucursales files were loaded.")

all_comercio_dfs = []
for file_path, file_source in comercio_files:
    try:
        # Specify 'id_comercio' and 'id_bandera' as string type to avoid DtypeWarning
        df = pd.read_csv(file_path, sep='|', dtype={'id_comercio': str, 'id_bandera': str}, low_memory=False)
        df['file_source'] = file_source
        all_comercio_dfs.append(df)
    except Exception as e:
        print(f"Error loading {file_path}: {e}")

if all_comercio_dfs:
    all_comercio_df = pd.concat(all_comercio_dfs, ignore_index=True)
    # Apply column cleaning
    comercio_prefixes = ['comercio_']
    all_comercio_df = clean_column_names(all_comercio_df, comercio_prefixes)
else:
    print("No comercio files were loaded.")

In [ ]:
bandera_filtro = ['Changomas', 'COTO CICSA', 'Disco', 'Express', 'HiperChangomas', 'Hipermercado Carrefour', 'Jumbo', 'Market', 'Maxi', 'SuperChangomas', 'Supermercados DIA','FARMACITY','SIMPLICITY']
my_location = ['-34.53535860586283', '-58.47559604248442']
sucursales = all_sucursales_df.drop(columns = [ 'lunes_horario_atencion', 'martes_horario_atencion', 'miercoles_horario_atencion', 'jueves_horario_atencion', 'viernes_horario_atencion', 'sabado_horario_atencion', 'domingo_horario_atencion'])
sucursales['distance'] = sucursales.apply(lambda row: distance_haversine(my_location[0], my_location[1], row['latitud'], row['longitud']), axis=1)
# all_sucursales_df[all_sucursales_df['bandera'].isin(bandera_filtro)]
sucursales_2 = sucursales[(sucursales['distance'] <= 6) | (sucursales['distance'].isna())]
# sucursales_3 = sucursales[(sucursales['id_comercio'] == "9") & (sucursales['id_sucursal'] == "5202")]
# sucursales_2

In [ ]:
# Define the list of specific product IDs to filter
headers = ["id_producto", "marca_ok", "segmento", "talle", "unidades"]

data = [
    [7500435228756, "Pampers", "Babydry", "P", 56],
    [7500435247368, "Pampers", "Babydry", "P", 36],
    [7500435237611, "Pampers", "Deluxe", "P", 36],
    # [, "Pampers", "Deluxe", "P", 56],
    [7500435228633, "Pampers", "Babysan", "P", 12],
    [7500435230506, "Pampers", "Babysan", "P", 52],
    [7790250042181, "Babysec", "Super Premium", "M", 48],
    # [7790250042211, "Babysec", "Super Premium", "P", 48],
    [7790250042792, "Babysec", "Premium", "M", 48],
    [7790250047230, "Babysec", "Premium", "M", 68],
    [7794626012969, "Huggies", "Flexi Confort", "M", 22], #no lo encuentra
    [7794626012839, "Huggies", "Flexi Confort", "M", 48],
    [7794626012853, "Huggies", "Flexi Confort", "M", 68],
    [7794626010019, "Huggies", "Supreme Care", "M", 22],
    [7794626013270, "Huggies", "Supreme Care", "M", 48],
    [7794626013317, "Huggies", "Supreme Care", "M", 68]
]

# Create a DataFrame from the product data
product_details_df = pd.DataFrame(data, columns=headers)

# Extract the specific_id_productos from the new DataFrame
specific_id_productos = product_details_df['id_producto'].tolist()

# Filter all_productos_df for these specific product IDs
filtered_productos_df_temp = all_productos_df[all_productos_df['id_producto'].isin(specific_id_productos)].copy()

# Merge with product_details_df to add marca_ok, segmento, talle, unidades
filtered_productos_df = pd.merge(
    filtered_productos_df_temp,
    product_details_df,
    on='id_producto',
    how='left'
)

print(f"Filtered {len(filtered_productos_df)} product entries with the specified IDs.")
# display(filtered_productos_df.head())

Filtered 7131 product entries with the specified IDs.


In [ ]:
# Merge the filtered products with the closer sucursales (sucursales_2)
# Using 'left' merge to keep all filtered product entries and add sucursales info where available
# Common columns for merging are 'id_comercio', 'id_bandera', 'id_sucursal'
final_merged_df = pd.merge(
    filtered_productos_df,
    sucursales_2,
    on=['id_comercio', 'id_bandera', 'id_sucursal'],
    how='left'
)

print("Merged filtered products with closer sucursales.")
# display(final_merged_df.head())

Merged filtered products with closer sucursales.


In [ ]:
merged_with_filtered_sucursales = pd.merge(
    filtered_productos_df,
    sucursales_2,
    on=['id_comercio', 'id_bandera', 'id_sucursal'],
    how='inner' # Use inner merge to only keep matching products and stores
)

final_merged_filtered_data = pd.merge(
    merged_with_filtered_sucursales,
    all_comercio_df,
    on=['id_comercio', 'id_bandera'],
    how='inner' # Use inner merge to only keep matching products, stores, and commercial entities
)

# Re-merge with product_details_df to ensure 'marca_ok', 'segmento', 'talle', 'unidades' are present
# This step is added to explicitly re-introduce these columns if they were lost during previous merges
final_merged_filtered_data = pd.merge(
    final_merged_filtered_data,
    product_details_df[['id_producto', 'marca_ok', 'segmento', 'talle', 'unidades']],
    on='id_producto',
    how='left' # Use left merge to keep all entries from the main DataFrame
)

print("Merged filtered products with specific sucursales and comercio (id_comercio = '10').")
# display(final_merged_filtered_data.head())

Merged filtered products with specific sucursales and comercio (id_comercio = '10').


In [ ]:
import numpy as np

selected_columns = [
    'id_producto',
    'descripcion',
    'cantidad_presentacion',
    'unidad_medida_presentacion',
    'marca',
    'precio_lista',
    'precio_referencia',
    'cantidad_referencia',
    'unidad_medida_referencia',
    'precio_unitario_promo1',
    'leyenda_promo1',
    'precio_unitario_promo2',
    'leyenda_promo2',
    'file_source_x',
    'nombre',
    'calle',
    'numero',
    'bandera_nombre',
    'marca_ok_y',
    'segmento_y',
    'talle_y',
    'unidades_y'
]

# Filter the DataFrame to keep only the selected columns
final_merged_filtered_data_selected = final_merged_filtered_data[selected_columns].copy()

# Ensure 'unidades_y' is numeric (important for division) and handle potential NaN values
final_merged_filtered_data_selected['unidades_y'] = pd.to_numeric(final_merged_filtered_data_selected['unidades_y'], errors='coerce')

# Calculate potential unit prices by dividing by 'unidades_y'
promo2_divided = final_merged_filtered_data_selected['precio_unitario_promo2'] / final_merged_filtered_data_selected['unidades_y']
promo1_divided = final_merged_filtered_data_selected['precio_unitario_promo1'] / final_merged_filtered_data_selected['unidades_y']
lista_divided = final_merged_filtered_data_selected['precio_lista'] / final_merged_filtered_data_selected['unidades_y']

# Replace infinite values (e.g., from division by zero) with NaN so fillna can work correctly
promo2_divided.replace([np.inf, -np.inf], np.nan, inplace=True)
promo1_divided.replace([np.inf, -np.inf], np.nan, inplace=True)
lista_divided.replace([np.inf, -np.inf], np.nan, inplace=True)

# Apply coalesce logic: use promo2_divided if available, else promo1_divided, else lista_divided
final_merged_filtered_data_selected['precio_unitario_por_unidad'] = (
    promo2_divided
    .fillna(promo1_divided)
    .fillna(lista_divided)
)

# Sort the DataFrame by the new calculated unit price
final_merged_filtered_data_selected = final_merged_filtered_data_selected.sort_values(by='precio_unitario_por_unidad', ascending=True)

print("DataFrame with selected columns, new calculated unit price, and sorted:")
display(final_merged_filtered_data_selected)

DataFrame with selected columns, new calculated unit price, and sorted:


,id_producto,descripcion,cantidad_presentacion,unidad_medida_presentacion,marca,precio_lista,precio_referencia,cantidad_referencia,unidad_medida_referencia,precio_unitario_promo1,...,file_source_x,nombre,calle,numero,bandera_nombre,marca_ok_y,segmento_y,talle_y,unidades_y,precio_unitario_por_unidad
0,7.790250e+12,PAÑAL PREMIUM SOFT M PAQ 48 UNI,48.0,uni,BABYSEC,19274.00,401.54,1.0,uni,NaN,...,2026-06-24/2026-06-24/sepa_1_comercio-sepa-12_...,FLORIDA,Av. San Martin,3029.0,COTO CICSA,Babysec,Premium,M,48,401.541667
1,7.790250e+12,PAÑAL PREMIUM SOFT M PAQ 48 UNI,48.0,uni,BABYSEC,19274.00,401.54,1.0,uni,NaN,...,2026-06-24/2026-06-24/sepa_1_comercio-sepa-12_...,MONROE,Av. Monroe,3284.0,COTO CICSA,Babysec,Premium,M,48,401.541667
2,7.790250e+12,PAÑAL PREMIUM SOFT M PAQ 48 UNI,48.0,uni,BABYSEC,19274.00,401.54,1.0,uni,NaN,...,2026-06-24/2026-06-24/sepa_1_comercio-sepa-12_...,CABILDO,Av. Cabildo,545.0,COTO CICSA,Babysec,Premium,M,48,401.541667
26,7.790250e+12,PAÑAL PREMIUM SOFT M PAQ 48 UNI,48.0,uni,BABYSEC,19274.00,401.54,1.0,uni,NaN,...,2026-06-24/2026-06-24/sepa_1_comercio-sepa-12_...,LIBERTADOR,Av. Libertador,6840.0,COTO CICSA,Babysec,Premium,M,48,401.541667
29,7.790250e+12,PAÑAL PREMIUM SOFT M PAQ 48 UNI,48.0,uni,BABYSEC,19274.00,401.54,1.0,uni,NaN,...,2026-06-24/2026-06-24/sepa_1_comercio-sepa-12_...,VTE LOPEZ,Av. Maipu,1758.0,COTO CICSA,Babysec,Premium,M,48,401.541667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54,7.794626e+12,PAÑAL HUGGIES SUPREME CARE HIPOALERGÉNICO T:M ...,68.0,uni,HUGGIES,56033.99,824.03,1.0,uni,NaN,...,2026-06-24/2026-06-24/sepa_1_comercio-sepa-12_...,CABILDO,Av. Cabildo,4125.0,COTO CICSA,Huggies,Supreme Care,M,68,824.029265
41,7.794626e+12,PAÑAL HUGGIES SUPREME CARE HIPOALERGÉNICO T:M ...,68.0,uni,HUGGIES,56033.99,824.03,1.0,uni,NaN,...,2026-06-24/2026-06-24/sepa_1_comercio-sepa-12_...,SAAVEDRA,Av. Ricardo Balbín,4027.0,COTO CICSA,Huggies,Supreme Care,M,68,824.029265
42,7.794626e+12,PAÑAL HUGGIES SUPREME CARE HIPOALERGÉNICO T:M ...,68.0,uni,HUGGIES,56033.99,824.03,1.0,uni,NaN,...,2026-06-24/2026-06-24/sepa_1_comercio-sepa-12_...,MUNRO,Av. Mitre,2951.0,COTO CICSA,Huggies,Supreme Care,M,68,824.029265
10,7.794626e+12,PAÑAL HUGGIES SUPREME CARE HIPOALERGÉNICO T:M ...,68.0,uni,HUGGIES,56033.99,824.03,1.0,uni,NaN,...,2026-06-24/2026-06-24/sepa_1_comercio-sepa-12_...,MAURE,Maure,1725.0,COTO CICSA,Huggies,Supreme Care,M,68,824.029265


In [ ]:
# Save the final merged DataFrame to a CSV file
output_csv_filename = 'filtered_products_with_nearby_stores.csv'
final_merged_filtered_data_selected.to_csv(output_csv_filename, index=False)

print(f"The merged data has been saved to '{output_csv_filename}'.")

The merged data has been saved to 'filtered_products_with_nearby_stores.csv'.


In [ ]:
import re # Import the regular expression module for more precise word matching

ean_to_check = 7500435228596

print(f"--- Details from product_details_df for EAN {ean_to_check} ---")
product_detail_row = product_details_df[product_details_df['id_producto'] == ean_to_check]
display(product_detail_row)

print(f"\n--- Entries from final_merged_filtered_data_selected for EAN {ean_to_check} ---")
filtered_data_for_ean = final_merged_filtered_data_selected[final_merged_filtered_data_selected['id_producto'] == ean_to_check]
display(filtered_data_for_ean[['id_producto', 'descripcion', 'talle_y', 'segmento_y', 'unidades_y']])


if not product_detail_row.empty:
    expected_talle = product_detail_row['talle'].iloc[0]
    expected_segmento = product_detail_row['segmento'].iloc[0]

    print(f"\nExpected 'Talle' from `product_details_df`: {expected_talle}")
    print(f"Expected 'Segmento' from `product_details_df`: {expected_segmento}")

    # Define common sizes for conflict detection (case-insensitive)
    # These are general size indicators that might appear as words in descriptions
    all_known_sizes_words = ['P', 'M', 'G', 'XG', 'XXG', 'RN', 'JUMBO', 'MEGA'] # Adding common aliases/words
    # Also consider full words like 'PEQUEÑO', 'GRANDE', 'MEDIANO', etc., if common in descriptions

    if not filtered_data_for_ean.empty:
        print("\n--- Cross-checking description with expected attributes and detecting conflicts ---")
        for idx, row in filtered_data_for_ean.iterrows():
            desc = row['descripcion']
            current_talle_y = row['talle_y']

            # Check if the expected talle is present as a whole word in the description
            # Use re.escape to handle special characters in 'expected_talle' if any
            talle_in_desc_match = re.search(r'\b' + re.escape(expected_talle) + r'\b', desc, re.IGNORECASE)
            talle_in_desc = bool(talle_in_desc_match)

            # Check for any conflicting sizes in the description
            conflicting_sizes_found = []
            for size_word in all_known_sizes_words:
                # Avoid matching the expected size against itself as a conflict
                if size_word.upper() == expected_talle.upper():
                    continue

                # Check if a conflicting size is present as a whole word
                if re.search(r'\b' + re.escape(size_word) + r'\b', desc, re.IGNORECASE):
                    conflicting_sizes_found.append(size_word)

            # Special handling for 'XGRA' as it's 'Extra Grande' and conflicts with 'P'
            if expected_talle.upper() == 'P' and re.search(r'\bXGRA\b', desc, re.IGNORECASE):
                if 'XGRA' not in conflicting_sizes_found: # Avoid duplicates if 'XGRA' is in all_known_sizes_words
                    conflicting_sizes_found.append('XGRA')

            print(f"Product Description: '{desc}'")
            print(f"  - Expected 'Talle' from `product_details_df`: '{expected_talle}'")
            print(f"  - `talle_y` in `final_merged_filtered_data_selected`: '{current_talle_y}'")
            print(f"  - Is expected 'Talle' ({expected_talle}) explicitly mentioned as a word in description? {talle_in_desc}")

            if conflicting_sizes_found:
                print(f"  - **Warning: Conflicting sizes found in description: {', '.join(conflicting_sizes_found)}**")

            if current_talle_y != expected_talle:
                print(f"  - **Discrepancy: `talle_y` in `final_merged_filtered_data_selected` ('{current_talle_y}') does not match expected from `product_details_df` ('{expected_talle}').**")
            elif not talle_in_desc and not conflicting_sizes_found:
                 print(f"  - Note: Expected 'Talle' ({expected_talle}) not explicitly found as a word in description, and no other sizes detected. The `talle_y` value comes from `product_details_df`.")
            elif not talle_in_desc and conflicting_sizes_found:
                 print(f"  - **Major Discrepancy: Expected 'Talle' ({expected_talle}) not explicitly found as a word in description, but conflicting sizes ({', '.join(conflicting_sizes_found)}) were found. The `talle_y` value comes from `product_details_df`.**")

    else:
        print(f"No entries found in `final_merged_filtered_data_selected` for EAN {ean_to_check}.")
else:
    print(f"No details found in `product_details_df` for EAN {ean_to_check}.")

--- Details from product_details_df for EAN 7500435228596 ---


,id_producto,marca_ok,segmento,talle,unidades



--- Entries from final_merged_filtered_data_selected for EAN 7500435228596 ---


,id_producto,descripcion,talle_y,segmento_y,unidades_y


No details found in `product_details_df` for EAN 7500435228596.


### Inferring Product Attributes from Grouped Descriptions

This section filters products by common brands (Huggies, Pampers, Babysec), aggregates all unique descriptions for each `id_producto`, and then attempts to infer `segmento` (product line), `talle` (size), and `unidades` (quantity) using regular expressions.

In [ ]:
import re

# 1. Filter all_productos_df for relevant brands
brand_keywords = ['huggies', 'pampers', 'babysec']

filtered_by_brand_df = all_productos_df[
    all_productos_df['descripcion'].fillna('').str.contains('|'.join(brand_keywords), case=False, na=False)
].copy()

# 2. Group by id_producto and aggregate descriptions
# Concatenate unique descriptions for each product
grouped_descriptions_df = filtered_by_brand_df.groupby('id_producto')['descripcion'].apply(lambda x: ' '.join(x.dropna().unique())).reset_index()
grouped_descriptions_df.rename(columns={'descripcion': 'all_descriptions'}, inplace=True)

# 3. Infer 'marca', 'segmento', 'talle', and 'unidades' from grouped descriptions

# Define brand-specific segment keywords
segment_keywords_by_brand = {
    'Pampers': {
        'Babydry': ['babydry', 'babydri'],
        'Babysan': ['babysan'],
        'Deluxe': ['deluxe'],
        'Active Sec': ['active sec'],
        'Confort Sec': ['confort sec'],
        'Premium Care': ['premium care', 'premium_care'],
        'Recien Nacido': ['recien nacido', 'newborn', 'rn']
    },
    'Babysec': {
        'Super Premium': ['super premium', 'superpremiun', 'super_premium'],
        'Premium': ['premium'],
        'Ultrasoft': ['ultrasoft']
    },
    'Huggies': {
        'Flexi Confort': ['flexi confort', 'flexi_confort'],
        'Supreme Care': ['supreme care', 'supreme_care'],
        'Natural Care': ['natural care', 'natural_care'],
        'Little Swimmers': ['little swimm', 'little swimmers']
    }
}

# Define all size keywords (from previous cells, plus some more variations)
all_size_keywords_extended = {
    'P': ['P', 'PEQUEÑO', 'PEQ', 'SMALL'],
    'M': ['M', 'MEDIANO', 'MED'],
    'G': ['G', 'GRANDE'],
    'XG': ['XG', 'EXTRA GRANDE', 'X-GRANDE'],
    'XXG': ['XXG', 'XX-GRANDE'],
    'RN': ['RN', 'RECIEN NACIDO', 'NEWBORN'],
    'JUMBO': ['JUMBO'],
    'MEGA': ['MEGA'],
    'SUPER': ['SUPER']
}

def infer_marca(text):
    text_upper = text.upper()
    for brand_name in brand_keywords:
        if re.search(r'\b' + re.escape(brand_name.upper()) + r'\b', text_upper):
            return brand_name.capitalize()
    return None

def infer_segmento(text, brand):
    text_upper = text.upper()
    if brand and brand in segment_keywords_by_brand:
        for segment, keywords in segment_keywords_by_brand[brand].items():
            for kw in keywords:
                if re.search(r'\b' + re.escape(kw.upper()) + r'\b', text_upper):
                    return segment
    # If brand is found but no specific segment, assign 'Others'
    if brand:
        return 'Others'
    return None

def infer_talle(text):
    text_upper = text.upper()
    # Prioritize specific sizes over general terms if both are present
    for talle, keywords in all_size_keywords_extended.items():
        for kw in keywords:
            if re.search(r'\b' + re.escape(kw.upper()) + r'\b', text_upper):
                return talle
    return None

def infer_unidades(text):
    text_upper = text.upper()

    # Prioritized patterns for unit extraction
    patterns = [
        r'\bX\s*(\d+)(?:\s*UNI|U)?\b',                                     # X 36, X36UNI, X36U
        r'\b(?:' + '|'.join(re.escape(k) for k in all_size_keywords_extended.keys()) + r')(\d+)\b', # MX34, XXG20
        r'\b(\d+)\s*PC\b',                                                  # 34PC
        r'(\d+)(?:\s*U|\s*UNI|\s*UNIDADES|\s*PAQ|\s*UN)\b',               # 48U, 48UNI, 48UN, 48 UNIDADES (with or without space)
        r'(\d+)\s*UN\b',                                                    # 48 UN (standalone)
        r'PAQ\s*(\d+)',                                                      # PAQ 48
        r'\b(\d+)\b$'                                                       # Standalone number at the end
    ]

    for pattern in patterns:
        match = re.search(pattern, text_upper)
        if match:
            return int(match.group(1))

    return None

grouped_descriptions_df['inferred_marca'] = grouped_descriptions_df['all_descriptions'].apply(infer_marca)

# Filter out products with 'TOA' in their descriptions (as requested previously)
grouped_descriptions_df = grouped_descriptions_df[~grouped_descriptions_df['all_descriptions'].str.contains(r'\bTOA\b', case=False, na=False)].copy()

# Filter out non-diaper products explicitly (e.g., wet wipes, liquid soap, shampoo, creams, towels)
# Using more flexible regex to catch variations like 'TOALL.HUMED.', 'TOALL HUMEDAS', 'JABON' standalone
non_diaper_keywords_regex = [
    r'TOALL(?:ITAS)?(?:\s*|\.|-)?HUMED[A]?S?', # Covers TOALLITAS HUMEDAS, TOALL.HUMED., TOALL HUMED, etc.
    r'JABON LIQUIDO',
    r'SHAMPOO',
    r'COLONIA',
    r'JABON\b',
    r'CREMA',
    r'TOALLAS'
]
non_diaper_pattern = '|'.join(non_diaper_keywords_regex)
grouped_descriptions_df = grouped_descriptions_df[~grouped_descriptions_df['all_descriptions'].str.contains(non_diaper_pattern, flags=re.IGNORECASE, na=False)].copy()

grouped_descriptions_df['inferred_segmento'] = grouped_descriptions_df.apply(lambda row: infer_segmento(row['all_descriptions'], row['inferred_marca']), axis=1)
grouped_descriptions_df['inferred_talle'] = grouped_descriptions_df['all_descriptions'].apply(infer_talle)
grouped_descriptions_df['inferred_unidades'] = grouped_descriptions_df['all_descriptions'].apply(infer_unidades)

# Display the resulting DataFrame
print("Products with grouped descriptions and inferred attributes:")
grouped_descriptions_df

Products with grouped descriptions and inferred attributes:


,id_producto,all_descriptions,inferred_marca,inferred_segmento,inferred_talle,inferred_unidades
1,7.500435e+12,PANAL PAMPERS REC.NACIDO RN+ 20 u.,Pampers,Recien Nacido,RN,20.0
4,7.500435e+12,"PAÑAL M PANTS PREMIUM CARE, PAMPERS, 34 cu PAÑ...",Pampers,Premium Care,M,34.0
5,7.500435e+12,"PAÑAL G PANTS PREMIUM CARE, PAMPERS, 30 cu PAÑ...",Pampers,Premium Care,G,30.0
6,7.500435e+12,"PAÑAL XG PANTS PREMIUM CARE, PAMPERS, 26 cu PA...",Pampers,Premium Care,XG,26.0
7,7.500435e+12,"PAÑAL XXG PANTS PREMIUM CARE, PAMPERS, 24 cu P...",Pampers,Premium Care,XXG,24.0
...,...,...,...,...,...,...
196,7.896008e+12,pañal HUGGIES little swimmers m 11u PCK-11.781...,Huggies,Little Swimmers,M,11.0
197,7.896008e+12,HUGGIES LITTLE SWIMM,Huggies,Little Swimmers,None,NaN
206,7.896062e+12,"PAÑAL PANTS G PREMIUM, BABYSEC, 28 cu BABYSEC ...",Babysec,Premium,G,NaN
207,7.896062e+12,"PAÑAL PANTS XG PREMIUM, BABYSEC, 22 cu BABYSEC...",Babysec,Premium,XG,NaN


In [ ]:
# Merge grouped_descriptions_df with product_details_df
# This will allow for a direct comparison between the inferred data and the provided EAN data
merged_inferred_and_provided_df = pd.merge(
    grouped_descriptions_df,
    product_details_df[['id_producto', 'marca_ok', 'segmento', 'talle', 'unidades']],
    on='id_producto',
    how='left', # Use inner merge to keep only products present in both DataFrames for direct comparison
    suffixes=('_inferred', '_provided')
)

print("Merged Inferred Attributes with Provided Product Details (for cross-checking):")
merged_inferred_and_provided_df

Merged Inferred Attributes with Provided Product Details (for cross-checking):


,id_producto,all_descriptions,inferred_marca,inferred_segmento,inferred_talle,inferred_unidades,marca_ok,segmento,talle,unidades
0,7.500435e+12,PANAL PAMPERS REC.NACIDO RN+ 20 u.,Pampers,Recien Nacido,RN,20.0,NaN,NaN,NaN,NaN
1,7.500435e+12,"PAÑAL M PANTS PREMIUM CARE, PAMPERS, 34 cu PAÑ...",Pampers,Premium Care,M,34.0,NaN,NaN,NaN,NaN
2,7.500435e+12,"PAÑAL G PANTS PREMIUM CARE, PAMPERS, 30 cu PAÑ...",Pampers,Premium Care,G,30.0,NaN,NaN,NaN,NaN
3,7.500435e+12,"PAÑAL XG PANTS PREMIUM CARE, PAMPERS, 26 cu PA...",Pampers,Premium Care,XG,26.0,NaN,NaN,NaN,NaN
4,7.500435e+12,"PAÑAL XXG PANTS PREMIUM CARE, PAMPERS, 24 cu P...",Pampers,Premium Care,XXG,24.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
171,7.896008e+12,pañal HUGGIES little swimmers m 11u PCK-11.781...,Huggies,Little Swimmers,M,11.0,NaN,NaN,NaN,NaN
172,7.896008e+12,HUGGIES LITTLE SWIMM,Huggies,Little Swimmers,None,NaN,NaN,NaN,NaN,NaN
173,7.896062e+12,"PAÑAL PANTS G PREMIUM, BABYSEC, 28 cu BABYSEC ...",Babysec,Premium,G,NaN,NaN,NaN,NaN,NaN
174,7.896062e+12,"PAÑAL PANTS XG PREMIUM, BABYSEC, 22 cu BABYSEC...",Babysec,Premium,XG,NaN,NaN,NaN,NaN,NaN


In [ ]:
merged_with_bandera_nombre = pd.merge(
    merged_inferred_and_provided_df,
    final_merged_filtered_data_selected[['id_producto', 'bandera_nombre']],
    on='id_producto',
    how='left'
)

# Aggregate bandera_nombre for each product, ensuring uniqueness
aggregated_bandera_df = merged_with_bandera_nombre.groupby(
    ['id_producto', 'all_descriptions', 'inferred_marca', 'inferred_segmento', 'inferred_talle', 'inferred_unidades',
     'marca_ok', 'segmento', 'talle', 'unidades']
)['bandera_nombre'].apply(lambda x: list(x.dropna().unique())).reset_index()

print("Merged Inferred Attributes with Provided Product Details and Aggregated Bandera Nombres:")
aggregated_bandera_df

Merged Inferred Attributes with Provided Product Details and Aggregated Bandera Nombres:


,id_producto,all_descriptions,inferred_marca,inferred_segmento,inferred_talle,inferred_unidades,marca_ok,segmento,talle,unidades,bandera_nombre
0,7.500435e+12,"PAÑAL P BABYDRY HIPER, PAMPERS, 56 cu PAÑAL PA...",Pampers,Babydry,P,56.0,Pampers,Babydry,P,56.0,"[COTO CICSA, Jumbo, Disco, Vea]"
1,7.500435e+12,PAÑAL PAMPERS BABYSAN HIPOALERG.T:P PAQ 52 UNI...,Pampers,Babysan,P,52.0,Pampers,Babysan,P,52.0,"[Supermercados DIA, COTO CICSA, Vea, Jumbo]"
2,7.500435e+12,"PAÑAL P DELUXE PROTECTION HIPER, PAMPERS, 36 c...",Pampers,Deluxe,P,36.0,Pampers,Deluxe,P,36.0,"[Market, Hipermercado Carrefour, Disco, Superm..."
3,7.790250e+12,"PAÑAL M SUPER PREMIUM HIPER, BABYSEC, 48 cu BA...",Babysec,Super Premium,M,48.0,Babysec,Super Premium,M,48.0,"[Jumbo, HiperChangomas]"
4,7.790250e+12,"PAÑAL M PREMIUM SOFT HIPER, BABYSEC, 48 cu PAN...",Babysec,Premium,M,48.0,Babysec,Premium,M,48.0,"[COTO CICSA, Hipermercado Carrefour, Market, H..."
5,7.794626e+12,"PAÑAL M FLEXI COMFORT HIPER, HUGGIES, 48 cu PA...",Huggies,Flexi Confort,M,2.0,Huggies,Flexi Confort,M,48.0,"[COTO CICSA, Vea, Disco, Jumbo, Supermercados ..."
6,7.794626e+12,"PAÑAL M FLEXI COMFORT AP, HUGGIES, 68 cu PAÑAL...",Huggies,Flexi Confort,M,68.0,Huggies,Flexi Confort,M,68.0,"[Hipermercado Carrefour, Market, Jumbo, Disco,..."
7,7.794626e+12,"PAÑAL M SUPREME CARE HIPER, HUGGIES, 48 cu HUG...",Huggies,Supreme Care,M,48.0,Huggies,Supreme Care,M,48.0,"[Disco, Jumbo, HiperChangomas]"
8,7.794626e+12,"PAÑAL M SUPREME CARE AP, HUGGIES, 68 cu PAÑAL ...",Huggies,Supreme Care,M,68.0,Huggies,Supreme Care,M,68.0,"[Jumbo, Hipermercado Carrefour, Market, COTO C..."


Empieza Nutrilon

In [ ]:
nutrilon_productos_df = all_productos_df[all_productos_df['descripcion'].str.contains('nutrilon', case=False, na=False)]


In [ ]:
nutrilon_productos_df

,id_comercio,id_bandera,id_sucursal,id_producto,ean,descripcion,cantidad_presentacion,unidad_medida_presentacion,marca,precio_lista,precio_referencia,cantidad_referencia,unidad_medida_referencia,precio_unitario_promo1,leyenda_promo1,precio_unitario_promo2,leyenda_promo2,file_source,leyenda_promo2
2097,2,1,004,7.795324e+12,1.0,"FORMULA LACTEA LV. ETAPA 1 PROFUT, NUTRILON, 2...",200.0,cm3,NUTRILON,3400.0,17000.00,1.0,lt,NaN,NaN,NaN,NaN,2026-06-24/2026-06-24/sepa_2_comercio-sepa-2_2...,NaN
2098,2,1,004,7.795324e+12,1.0,"FORMULA LACTEA LV. ETAPA 2 PROFUT, NUTRILON, 2...",200.0,cm3,NUTRILON,3250.0,16250.00,1.0,lt,NaN,NaN,NaN,NaN,2026-06-24/2026-06-24/sepa_2_comercio-sepa-2_2...,NaN
2099,2,1,004,7.795324e+12,1.0,"LECHE LV MODIFICADA ETAPA 3 PROFUT, NUTRILON, ...",200.0,cm3,NUTRILON,3100.0,15500.00,1.0,lt,NaN,NaN,NaN,NaN,2026-06-24/2026-06-24/sepa_2_comercio-sepa-2_2...,NaN
6212,2,1,006,7.795323e+12,1.0,"FORMULA LACTEA POLVO PROFUTURA ET1, NUTRILON, ...",800.0,gr,NUTRILON,58050.0,72562.50,1.0,kgr,NaN,NaN,NaN,NaN,2026-06-24/2026-06-24/sepa_2_comercio-sepa-2_2...,NaN
6213,2,1,006,7.795323e+12,1.0,"FORMULA LACTEA POLVO PROFUTURA ET2, NUTRILON, ...",800.0,gr,NUTRILON,52600.0,65750.00,1.0,kgr,NaN,NaN,NaN,NaN,2026-06-24/2026-06-24/sepa_2_comercio-sepa-2_2...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14085720,9,2,48,7.795324e+12,1.0,"leche en Polvo Nutrilon 3 Pouch 1,2 kg PCH-120...",1.2,gr.,NUTRILON,62750.0,5229166.67,100.0,gr.,NaN,NaN,NaN,NaN,2026-06-24/2026-06-24/sepa_1_comercio-sepa-9_2...,NaN
14085721,9,2,33,7.795324e+12,1.0,"leche en Polvo Nutrilon 3 Pouch 1,2 kg PCH-120...",1.2,gr.,NUTRILON,62750.0,5229166.67,100.0,gr.,NaN,NaN,NaN,NaN,2026-06-24/2026-06-24/sepa_1_comercio-sepa-9_2...,NaN
14085722,9,3,138,7.795324e+12,1.0,"leche en Polvo Nutrilon 3 Pouch 1,2 kg PCH-120...",1.2,gr.,NUTRILON,62750.0,5229166.67,100.0,gr.,NaN,NaN,NaN,NaN,2026-06-24/2026-06-24/sepa_1_comercio-sepa-9_2...,NaN
14085723,9,2,740,7.795324e+12,1.0,"leche en Polvo Nutrilon 3 Pouch 1,2 kg PCH-120...",1.2,gr.,NUTRILON,62750.0,5229166.67,100.0,gr.,NaN,NaN,NaN,NaN,2026-06-24/2026-06-24/sepa_1_comercio-sepa-9_2...,NaN


In [ ]:
import re

def infer_state(description):
    desc_lower = str(description).lower()
    if re.search(r'\b(\d+)\s*(ml|cc)\b', desc_lower): # Check for ml or cc first
        return 'liquido'
    if re.search(r'\b(\d+)\s*(gr|g)\b', desc_lower): # Check for gr or g first
        return 'polvo'
    if re.search(r'\b(liquido|fluido)\b', desc_lower):
        return 'liquido'
    if re.search(r'\b(polvo|en polvo)\b', desc_lower):
        return 'polvo'
    return 'otro'

def infer_etapa(description):
    desc_lower = str(description).lower()
    if re.search(r'\b0\s*a\s*6\s*meses\b', desc_lower): # '0 A 6 MESES' for Etapa 1
        return 1
    match = re.search(r'etapa\s*(\d)|et\s*(\d)|\b(\d)\b|pf(\d)', desc_lower) # Handles 'etapa 1', 'et 1', ' 1 ', 'PF1'
    if match:
        # Prioritize 'etapa', then 'et', then standalone digit, then 'pf' digit
        if match.group(1): return int(match.group(1))
        if match.group(2): return int(match.group(2))
        if match.group(3): return int(match.group(3))
        if match.group(4): return int(match.group(4))
    return None

def infer_weight(description):
    desc_lower = str(description).lower()
    # Pattern for ml or cc
    ml_match = re.search(r'(\d+)\s*(ml|cc)\b', desc_lower)
    if ml_match:
        return f"{ml_match.group(1)}{ml_match.group(2)}"

    # Pattern for kg (including 1.2 kg)
    kg_match = re.search(r'(\d+\s*,\s*\d+|\d+\.\d+|\d+)\s*(kg)\b', desc_lower)
    if kg_match:
        # Replace comma with dot for consistent float conversion if necessary
        weight_val = kg_match.group(1).replace(',', '.')
        return f"{float(weight_val)}{kg_match.group(2)}"

    # Pattern for gr (including 800gr explicitly and '800' interpreted as grams)
    gr_match = re.search(r'(\d+)\s*(gr|g)\b', desc_lower)
    if gr_match:
        return f"{gr_match.group(1)}{gr_match.group(2)}"
    # If '800' appears without a unit and no other weight was found, assume '800gr'
    if re.search(r'\b800\b', desc_lower) and not (ml_match or kg_match or gr_match):
        return '800gr'

    return None

def infer_segment(description):
    desc_lower = str(description).lower()
    # Prioritize more specific patterns first
    if re.search(r'\bprofutura\b', desc_lower): # Full 'profutura'
        return 'profutura'
    if re.search(r'\bprofut\b', desc_lower): # 'profut' prefix
        return 'profutura'
    if re.search(r'\b(pro|pf|profu)\b', desc_lower): # 'PRO', 'PF', 'PROFU'
        return 'profutura'
    if re.search(r'\bcomfort\b', desc_lower):
        return 'comfort'
    if re.search(r'\badvance\b', desc_lower):
        return 'advance'
    return 'otros'

# Apply the inference functions to nutrilon_productos_df
nutrilon_productos_inferred_df = nutrilon_productos_df.copy()
nutrilon_productos_inferred_df['estado'] = nutrilon_productos_inferred_df['descripcion'].apply(infer_state)
nutrilon_productos_inferred_df['etapa'] = nutrilon_productos_inferred_df['descripcion'].apply(infer_etapa)
nutrilon_productos_inferred_df['peso'] = nutrilon_productos_inferred_df['descripcion'].apply(infer_weight)
nutrilon_productos_inferred_df['segmento_nutrilon'] = nutrilon_productos_inferred_df['descripcion'].apply(infer_segment)

# Group by id_producto and descripcion, and show the inferred attributes
inferred_summary_df = nutrilon_productos_inferred_df.groupby(['id_producto', 'descripcion']).agg(
    estado=('estado', 'first'),
    etapa=('etapa', 'first'),
    peso=('peso', 'first'),
    segmento_nutrilon=('segmento_nutrilon', 'first')
).reset_index()

print("Nutrilon products with refined inferred attributes:")
display(inferred_summary_df)

Nutrilon products with refined inferred attributes:


,id_producto,descripcion,estado,etapa,peso,segmento_nutrilon
0,7.795323e+12,LECHE EN POLVO NUTRILON PRO 1 LATA 800gr.,polvo,1.0,800gr,profutura
1,7.795323e+12,LECHE EN POLVO NUTRILON PRO 2 LATA 800gr.,polvo,2.0,800gr,profutura
2,7.795323e+12,LECHE EN POLVO NUTRILON PRO 3 LATA 800gr.,polvo,3.0,800gr,profutura
3,7.795323e+12,NUTRILON PREMIUM 1 0 A 6 MESES 200ml.,liquido,1.0,200ml,otros
4,7.795323e+12,NUTRILON PRO FUTURA 2 6-12 MESES 200ml.,liquido,2.0,200ml,profutura
...,...,...,...,...,...,...
86,7.795324e+12,"NUTRILON 3 POUCH POR 1,2 KG",otro,3.0,1.2kg,otros
87,7.795324e+12,"leche en Polvo Nutrilon 3 Pouch 1,2 kg PCH-120...",polvo,3.0,1.2kg,otros
88,7.795324e+12,LECHE E/POLVO PROFUTURA 4 DE 2 AÑOS ADELA.NUTR...,polvo,4.0,1.2kg,profutura
89,7.795324e+12,LECHE POLVO INFANTIL ET 4 NUTRILON POUCH X 1.2 KG,polvo,4.0,1.2kg,otros


In [ ]:
inferred_summary_df

,id_producto,descripcion,estado,etapa,peso,segmento_nutrilon
0,7.795323e+12,LECHE EN POLVO NUTRILON PRO 1 LATA 800gr.,polvo,NaN,800gr,otros
1,7.795323e+12,LECHE EN POLVO NUTRILON PRO 2 LATA 800gr.,polvo,NaN,800gr,otros
2,7.795323e+12,LECHE EN POLVO NUTRILON PRO 3 LATA 800gr.,polvo,NaN,800gr,otros
3,7.795323e+12,NUTRILON PREMIUM 1 0 A 6 MESES 200ml.,otro,NaN,200ml,otros
4,7.795323e+12,NUTRILON PRO FUTURA 2 6-12 MESES 200ml.,otro,NaN,200ml,otros
...,...,...,...,...,...,...
86,7.795324e+12,"NUTRILON 3 POUCH POR 1,2 KG",otro,NaN,1.2kg,otros
87,7.795324e+12,"leche en Polvo Nutrilon 3 Pouch 1,2 kg PCH-120...",polvo,NaN,1.2kg,otros
88,7.795324e+12,LECHE E/POLVO PROFUTURA 4 DE 2 AÑOS ADELA.NUTR...,polvo,NaN,1.2kg,profutura
89,7.795324e+12,LECHE POLVO INFANTIL ET 4 NUTRILON POUCH X 1.2 KG,polvo,4.0,1.2kg,otros


In [ ]:
# 1. Group by id_producto and aggregate all unique descriptions
nutrilon_grouped_descriptions_df = nutrilon_productos_df.groupby('id_producto')['descripcion'].apply(lambda x: ' '.join(x.dropna().unique())).reset_index()
nutrilon_grouped_descriptions_df.rename(columns={'descripcion': 'all_descriptions'}, inplace=True)

# 2. Apply the inference functions to the aggregated descriptions
nutrilon_grouped_descriptions_df['estado'] = nutrilon_grouped_descriptions_df['all_descriptions'].apply(infer_state)
nutrilon_grouped_descriptions_df['etapa'] = nutrilon_grouped_descriptions_df['all_descriptions'].apply(infer_etapa)
nutrilon_grouped_descriptions_df['peso'] = nutrilon_grouped_descriptions_df['all_descriptions'].apply(infer_weight)
nutrilon_grouped_descriptions_df['segmento_nutrilon'] = nutrilon_grouped_descriptions_df['all_descriptions'].apply(infer_segment)

print("Nutrilon products with unique id_producto and inferred attributes from grouped descriptions:")
nutrilon_grouped_descriptions_df

Nutrilon products with unique id_producto and inferred attributes from grouped descriptions:


,id_producto,all_descriptions,estado,etapa,peso,segmento_nutrilon
0,7.795323e+12,LECHE EN POLVO NUTRILON PRO 1 LATA 800gr.,polvo,1.0,800gr,profutura
1,7.795323e+12,LECHE EN POLVO NUTRILON PRO 2 LATA 800gr.,polvo,2.0,800gr,profutura
2,7.795323e+12,LECHE EN POLVO NUTRILON PRO 3 LATA 800gr.,polvo,3.0,800gr,profutura
3,7.795323e+12,NUTRILON PREMIUM 1 0 A 6 MESES 200ml.,liquido,1.0,200ml,otros
4,7.795323e+12,NUTRILON PRO FUTURA 2 6-12 MESES 200ml.,liquido,2.0,200ml,profutura
5,7.795323e+12,NUTRILON PROFUTURA 3 200ml.,liquido,3.0,200ml,profutura
6,7.795323e+12,"FORMULA LACTEA POLVO PROFUTURA ET1, NUTRILON, ...",polvo,1.0,800gr,profutura
7,7.795323e+12,"FORMULA LACTEA POLVO PROFUTURA ET2, NUTRILON, ...",polvo,2.0,800gr,profutura
8,7.795323e+12,"LECHE EN POLVO MODIF. PROFUTURA ET3, NUTRILON,...",polvo,3.0,800gr,profutura
9,7.795323e+12,"LECHE MODIF EN POLVO PRO FUTURA E4, NUTRILON, ...",polvo,4.0,800gr,profutura


In [ ]:
selected_nutrilons = [
    [7795323000785, "polvo", 2, 0.8],
[7795323000846, "liquido", 2, 0.2],
[7795323002437, "polvo", 2, 0.8],
[7795323002475, "liquido", 2, 0.2],
[7795323772293, "polvo", 2, 0.8],
[7795323772620, "liquido", 2, 0.2],
[7795323775331, "liquido", 2, 0.2],
[7795323775393, "polvo", 2, 1.2]
]

In [ ]:
# 1. Create a DataFrame from selected_nutrilons
nutrilon_headers = ['id_producto', 'estado_target', 'etapa_target', 'peso_target_value']
nutrilon_target_products_df = pd.DataFrame(selected_nutrilons, columns=nutrilon_headers)

# Helper function to determine base unit quantity for price comparison
def get_base_unit_quantity(row):
    estado = str(row['estado_target']).lower()
    peso_value = row['peso_target_value']
    if estado == 'polvo': # Assume kg to grams
        return peso_value * 1000  # Convert kg to grams
    elif estado == 'liquido': # Assume liters to ml
        return peso_value * 1000  # Convert liters to ml
    return None

# Add base_unit_quantity column
nutrilon_target_products_df['base_unit_quantity'] = nutrilon_target_products_df.apply(get_base_unit_quantity, axis=1)

print("Nutrilon Target Products DataFrame:")
display(nutrilon_target_products_df)

Nutrilon Target Products DataFrame:


,id_producto,estado_target,etapa_target,peso_target_value,base_unit_quantity
0,7795323000785,polvo,2,0.8,800.0
1,7795323000846,liquido,2,0.2,200.0
2,7795323002437,polvo,2,0.8,800.0
3,7795323002475,liquido,2,0.2,200.0
4,7795323772293,polvo,2,0.8,800.0
5,7795323772620,liquido,2,0.2,200.0
6,7795323775331,liquido,2,0.2,200.0
7,7795323775393,polvo,2,1.2,1200.0


In [ ]:
# 2. Filter all_productos_df for these specific Nutrilon product IDs
specific_nutrilon_id_productos = nutrilon_target_products_df['id_producto'].tolist()

filtered_nutrilon_productos_df_temp = all_productos_df[
    all_productos_df['id_producto'].isin(specific_nutrilon_id_productos)
].copy()

# Merge with nutrilon_target_products_df to add the target attributes (estado, etapa, peso)
filtered_nutrilon_products_with_targets = pd.merge(
    filtered_nutrilon_productos_df_temp,
    nutrilon_target_products_df,
    on='id_producto',
    how='left'
)

# Merge with sucursales_2 (closer stores)
merged_nutrilon_with_sucursales = pd.merge(
    filtered_nutrilon_products_with_targets,
    sucursales_2,
    on=['id_comercio', 'id_bandera', 'id_sucursal'],
    how='inner' # Only keep products found in closer stores
)

# Merge with all_comercio_df (commercial entity details)
final_merged_nutrilon_df = pd.merge(
    merged_nutrilon_with_sucursales,
    all_comercio_df,
    on=['id_comercio', 'id_bandera'],
    how='inner'
)

print(f"Found {len(final_merged_nutrilon_df)} entries for selected Nutrilon products merged with store and commercial data.")
display(final_merged_nutrilon_df.head())

Found 239 entries for selected Nutrilon products merged with store and commercial data.


,id_comercio,id_bandera,id_sucursal,id_producto,ean,descripcion,cantidad_presentacion,unidad_medida_presentacion,marca,precio_lista,...,domingo_horario_atencion,distance,cuit,razon_social,bandera_nombre,bandera_url,ultima_actualizacion,version_sepa,file_source,version_sepa
0,12,1,170,7.795324e+12,1.0,LECHE INF.1 PROFUTURA 6 A 12 MESES NUTRILON TT...,200.0,ml,NUTRILON,3110.0,...,NaN,2.473333,3.054808e+10,COTO CENTRO INTEGRAL DE COMERCIALIZACION S.A.,COTO CICSA,http://www.coto.com.ar/,2026-06-24T01:06:14-03:00,1.0,2026-06-24/2026-06-24/sepa_1_comercio-sepa-12_...,NaN
1,12,1,92,7.795324e+12,1.0,LECHE INF.1 PROFUTURA 6 A 12 MESES NUTRILON TT...,200.0,ml,NUTRILON,3110.0,...,NaN,1.758494,3.054808e+10,COTO CENTRO INTEGRAL DE COMERCIALIZACION S.A.,COTO CICSA,http://www.coto.com.ar/,2026-06-24T01:06:14-03:00,1.0,2026-06-24/2026-06-24/sepa_1_comercio-sepa-12_...,NaN
2,12,1,22,7.795323e+12,1.0,LECHE EN POLVO PRO FUTURA NUTRILON 2 LAT 800 GRM,800.0,grm,NUTRILON,58065.0,...,NaN,4.347905,3.054808e+10,COTO CENTRO INTEGRAL DE COMERCIALIZACION S.A.,COTO CICSA,http://www.coto.com.ar/,2026-06-24T01:06:14-03:00,1.0,2026-06-24/2026-06-24/sepa_1_comercio-sepa-12_...,NaN
3,12,1,181,7.795324e+12,1.0,LECHE INF.1 PROFUTURA 6 A 12 MESES NUTRILON TT...,200.0,ml,NUTRILON,3110.0,...,NaN,2.412676,3.054808e+10,COTO CENTRO INTEGRAL DE COMERCIALIZACION S.A.,COTO CICSA,http://www.coto.com.ar/,2026-06-24T01:06:14-03:00,1.0,2026-06-24/2026-06-24/sepa_1_comercio-sepa-12_...,NaN
4,12,1,188,7.795324e+12,1.0,LECHE INF.1 PROFUTURA 6 A 12 MESES NUTRILON TT...,200.0,ml,NUTRILON,3110.0,...,NaN,4.332315,3.054808e+10,COTO CENTRO INTEGRAL DE COMERCIALIZACION S.A.,COTO CICSA,http://www.coto.com.ar/,2026-06-24T01:06:14-03:00,1.0,2026-06-24/2026-06-24/sepa_1_comercio-sepa-12_...,NaN


In [ ]:
# 3. Select relevant columns and calculate unit price for Nutrilon products

selected_columns_nutrilon = [
    'id_producto',
    'descripcion',
    'marca',
    'precio_lista',
    'precio_referencia',
    'precio_unitario_promo1',
    'leyenda_promo1',
    'precio_unitario_promo2',
    'leyenda_promo2',
    'estado_target',
    'etapa_target',
    'peso_target_value',
    'base_unit_quantity',
    'file_source_x',
    'nombre',
    'calle',
    'numero',
    'bandera_nombre'
]

# Filter the DataFrame to keep only the selected columns
final_merged_nutrilon_selected = final_merged_nutrilon_df[selected_columns_nutrilon].copy()

# Ensure 'base_unit_quantity' is numeric and handle potential NaN values
final_merged_nutrilon_selected['base_unit_quantity'] = pd.to_numeric(final_merged_nutrilon_selected['base_unit_quantity'], errors='coerce')

# Calculate potential unit prices by dividing by 'base_unit_quantity'
promo2_divided_nutrilon = final_merged_nutrilon_selected['precio_unitario_promo2'] / final_merged_nutrilon_selected['base_unit_quantity']
promo1_divided_nutrilon = final_merged_nutrilon_selected['precio_unitario_promo1'] / final_merged_nutrilon_selected['base_unit_quantity']
lista_divided_nutrilon = final_merged_nutrilon_selected['precio_lista'] / final_merged_nutrilon_selected['base_unit_quantity']

# Replace infinite values (e.g., from division by zero) with NaN so fillna can work correctly
promo2_divided_nutrilon.replace([np.inf, -np.inf], np.nan, inplace=True)
promo1_divided_nutrilon.replace([np.inf, -np.inf], np.nan, inplace=True)
lista_divided_nutrilon.replace([np.inf, -np.inf], np.nan, inplace=True)

# Apply coalesce logic: use promo2_divided if available, else promo1_divided, else lista_divided
final_merged_nutrilon_selected['precio_unitario_por_base_unit'] = (
    promo2_divided_nutrilon
    .fillna(promo1_divided_nutrilon)
    .fillna(lista_divided_nutrilon)
)

# Sort the DataFrame by the new calculated unit price to find the cheapest
cheapest_nutrilon_by_ean = final_merged_nutrilon_selected.sort_values(by=['id_producto', 'precio_unitario_por_base_unit'], ascending=True)

print("Cheapest Nutrilon products by EAN (unit price):")
display(cheapest_nutrilon_by_ean[['id_producto', 'descripcion', 'estado_target', 'etapa_target', 'peso_target_value', 'base_unit_quantity', 'precio_unitario_por_base_unit', 'bandera_nombre']].head(10))

Cheapest Nutrilon products by EAN (unit price):


,id_producto,descripcion,estado_target,etapa_target,peso_target_value,base_unit_quantity,precio_unitario_por_base_unit,bandera_nombre
35,7.795323e+12,LECHE POLVO INFANTIL ET 2 NUTRILON TARRO X 800...,polvo,2,0.8,800.0,68.87375,Hipermercado Carrefour
36,7.795323e+12,LECHE POLVO INFANTIL ET 2 NUTRILON TARRO X 800...,polvo,2,0.8,800.0,68.87375,Market
42,7.795323e+12,LECHE POLVO INFANTIL ET 2 NUTRILON TARRO X 800...,polvo,2,0.8,800.0,68.87375,Market
45,7.795323e+12,LECHE POLVO INFANTIL ET 2 NUTRILON TARRO X 800...,polvo,2,0.8,800.0,68.87375,Market
46,7.795323e+12,LECHE POLVO INFANTIL ET 2 NUTRILON TARRO X 800...,polvo,2,0.8,800.0,68.87375,Market
49,7.795323e+12,LECHE POLVO INFANTIL ET 2 NUTRILON TARRO X 800...,polvo,2,0.8,800.0,68.87375,Market
50,7.795323e+12,LECHE POLVO INFANTIL ET 2 NUTRILON TARRO X 800...,polvo,2,0.8,800.0,68.87375,Market
66,7.795323e+12,LECHE POLVO INFANTIL ET 2 NUTRILON TARRO X 800...,polvo,2,0.8,800.0,68.87375,Hipermercado Carrefour
77,7.795323e+12,LECHE POLVO INFANTIL ET 2 NUTRILON TARRO X 800...,polvo,2,0.8,800.0,68.87375,Market
78,7.795323e+12,LECHE POLVO INFANTIL ET 2 NUTRILON TARRO X 800...,polvo,2,0.8,800.0,68.87375,Market
